# 기존 M5 설계 재실험: 단일 음성 BPR, Dunnhumby 10 seeds

기존 10시드 실험의 M1·M2·M4·M5 구조, 데이터 분할, 100 epoch, 임베딩 차원, 학습률, rho=0.15, lambda=0.5를 유지합니다. 네 arm 모두에서 K=5 음성 평균만 LightGCN 원 논문의 핵심 BPR 표본구성과 같은 양성 1개–균등 음성 1개 비교로 바꿉니다. seed 42를 포함한 42~51을 모두 새로 학습합니다. 이미 노출된 DAY 698~704 test의 사후 손실설정 정정 실행이므로, 결과를 다시 보고 모형이나 하이퍼파라미터를 조정하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '3330346d43fcbdb3ada54ca58f8e69f5483969cf'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(['git', 'clone', REPO_URL, str(repo)], text=True, capture_output=True)
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)


In [ ]:
import json
import torch
from lightgcn_clv_m5_nv_economic_positive_weight_k1 import MODEL_IDS
from lightgcn_clv_m5_nv_economic_positive_weight_k1_test import (
    FULL_SEEDS,
    configure_m5_nv_economic_positive_k1_test_run,
    preflight_summary,
    run_m5_nv_economic_positive_k1_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_nv_economic_positive_k1_test_run(
    seeds=FULL_SEEDS,
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_explicit_nv_personalized_positive_weight_single_negative_bpr_test10_v1',
)
summary = preflight_summary(cfg)
assert cfg.seeds == tuple(range(42, 52))
assert cfg.negative_count == 1
assert len(MODEL_IDS) == 4
assert len(summary['models']) == 4
assert summary['unchanged_from_prior_ten_seed_run']['validation_constructed'] is False
assert summary['unchanged_from_prior_ten_seed_run']['holdout_constructed'] is False
assert summary['task_guards']['new_item_task'] is True
assert 'post-hoc' in summary['protocol_status']
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = run_m5_nv_economic_positive_k1_test(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) seed별 test 절대지표')
show(result_df)
print('2) 10시드 평균·표준편차·95% t 구간')
show(result_df.attrs['mean'])
print('3) seed별 M1·M4 기준 전체 지표 비교')
show(result_df.attrs['comparison'])
print('4) 동일 seed 대응차 평균과 개선 seed 수')
show(result_df.attrs['paired_mean'])
print('5) M2 × M4 상호작용 — 기술적 보조 결과')
show(result_df.attrs['interaction'])
print('6) seed별 기술적 판독')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
